# NN Benchmark: Does the Trend Component Behave Like CTAs?

The replicated neural network (NN) splits each market's Managed Money (MM) position into a **trend component**, which the paper reads as CTA positioning, and a **time-varying bias**, which it reads as discretionary positioning. Nothing in the COT data confirms that split. Here we check it against an independent source: the daily returns of the largest trend-following CTAs in the **SG Trend Index**. The NN is trained exactly as in the paper; SG returns are used only to evaluate its positions.

**The trend component earns CTA-like P&L; the bias does not, in a robust sense.** Weekly energy P&L on the NN's trend positions correlates **0.35** with SG Trend Index returns (rank correlation 0.32; t = 8.0 in a joint regression). That is close to the 20/120 rule behind the SG Trend Indicator (0.38). Bias P&L reaches 0.20, but only **0.06** by rank, so a few large weeks drive it. The link weakens in 2023–2025.

This is one of three SG notebooks, meant to be read in order:

1. [`nn_benchmark.ipynb`](nn_benchmark.ipynb): does the NN's trend component earn CTA-like P&L? SG returns are used only as a **benchmark**.
2. [`predict_sg_index.ipynb`](predict_sg_index.ipynb): the paper's momentum layer with SG returns as the **training target**.
3. [`sg_input_feature.ipynb`](sg_input_feature.ipynb): SG-derived **input features** as corrections to the MM nowcast.

Helpers are in [`src/sg_cta.py`](src/sg_cta.py).

## Setup

In [3]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'working_ideas' else Path.cwd()
PROJECT_ROOT = REPO_ROOT / 'paper_replication'
sys.path[:0] = [str(PROJECT_ROOT / 'src'), str(REPO_ROOT / 'working_ideas' / 'src')]

import torch
from paper_replication.config import HORIZONS, ReplicationConfig
from paper_replication.evaluate import run_replication
import sg_cta as sg
from sg_cta import KEYS, MARKETS, PNL_LABELS

OUTPUT = PROJECT_ROOT / 'outputs' / 'nn_benchmark'
OUTPUT.mkdir(parents=True, exist_ok=True)
torch.set_num_threads(1)
config = ReplicationConfig(data_dir=PROJECT_ROOT / 'data_required', basis='futures_only',
                           train_weeks=104, seed=7)
COLORS = {'trend': '#277f8e', 'bias': '#b7782c', 'rule': '#96587e', 'sg': '#596eaf'}
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

## Data

We refit the original NN on the same rolling 104-week windows as the replication, and record the two parts of each fitted position:

$$y_{m,t}\approx \underbrace{T_{m,t}}_{\text{trend}}+\underbrace{b_{m,t}}_{\text{bias}},\qquad MM_{m,t}=\mu_{m,t}+\sigma_{m,t}\,y_{m,t}.$$

$\mu$ and $\sigma$ are the lagged one-year mean and scale used to normalize positions. `run_replication` computes this trend/bias split directly from the model it already fits each window, so `mm_std_lag * (trend_cur - trend_prev)` equals the baseline's `predicted_change` by construction, not by a second refit.

In [ ]:
reference = run_replication(config, progress=False)
baseline = reference['predictions'].copy()
panel = reference['weekly_panel']
daily = sg.add_trend_rule(reference['daily_features'])
sg_levels = sg.load_sg_log_levels(config.data_dir)

decomposition, mm_loadings = sg.run_decomposition(panel, config)
check = baseline.merge(decomposition, on=KEYS, validate='one_to_one')
gap = (check['mm_std_lag'] * (check['trend_cur'] - check['trend_prev']) - check['predicted_change']).abs().max()
assert gap < 1, gap
print(f'{len(check):,} market-weeks; largest gap to the baseline predicted change: {gap:.3f} contracts')

## Does the trend component earn CTA-like P&L?

If the trend component is CTA positioning, holding it should earn money in the same weeks as the largest CTAs. We split last week's MM position into three parts, all in contracts:

$$MM_{m,t-1}=\underbrace{\sigma_{m,t-1}T_{m,t-1}}_{\text{trend}}+\underbrace{\mu_{m,t-1}+\sigma_{m,t-1}b_{m,t-1}}_{\text{bias}}+\underbrace{e_{m,t-1}}_{\text{fit residual}}$$

where

| Symbol | Meaning | Column |
|---|---|---|
| $m$, $t$ | Market and COT reporting week; $t-1$ is the previous report | `market`, `report_week_tuesday` |
| $MM_{m,t-1}$ | Futures-only net Managed Money position, in contracts | `mm_net` |
| $\mu_{m,t-1}$ | One-year (252 trading dates) rolling mean of $MM$, lagged one report | `mm_mean_lag` |
| $\sigma_{m,t-1}$ | One-year rolling scale of $MM$ around that mean, lagged one report; converts normalized units to contracts | `mm_std_lag` |
| $T_{m,t-1}$ | NN trend component in normalized units, $\sum_k W_m(k)\,R\big(\sum_n w_k(n)\,x_{m,t-1}(n)\big)$: momentum inputs $x$, shared horizon weights $w_k$, reaction function $R$, market weights $W_m$ | `trend_prev` |
| $b_{m,t-1}$ | NN time-varying bias in normalized units, at the last training week | `bias_prev` |
| $e_{m,t-1}$ | Fit residual in contracts, $MM_{m,t-1}-\sigma_{m,t-1}T_{m,t-1}-\mu_{m,t-1}-\sigma_{m,t-1}b_{m,t-1}$ | `q_residual` |

$T$ and $b$ come from the network fitted for week $t$, trained on the 104 complete weeks ending at $t-1$.

Each part's P&L this week is its contracts × contract multiplier × price change on the held contract, summed over the six markets. The positions use MM data up to $t-1$ only, which is published before week $t$ ends, and no week-$t$ return. We then compare each P&L series with the week's SG return, sampled on the same COT as-of dates.

As a benchmark we add the rule behind the **SG Trend Indicator** (20/120-day moving-average crossover, always in the market, volatility-scaled), applied to our six markets.

In [ ]:
pnl = sg.weekly_energy_pnl(panel, daily, decomposition, sg_levels)
print(f"{len(pnl)} weeks, {pnl.index.min():%Y-%m-%d} to {pnl.index.max():%Y-%m-%d}")
correlations = sg.pnl_correlations(pnl, 'SG Trend')
display(correlations.round(3))

Next we regress the SG return jointly on the three standardized P&L series, with Newey–West t-statistics (4 lags), for each SG series and for SG Trend by sub-period.

In [ ]:
attribution = sg.attribution_table(pnl, {
    'SG Trend': ('SG Trend', 2015, 2025), 'SG CTA': ('SG CTA', 2015, 2025),
    'SG Trend Indicator': ('SG Trend Indicator', 2015, 2025),
    'SG Trend, 2015–2019': ('SG Trend', 2015, 2019), 'SG Trend, 2020–2022': ('SG Trend', 2020, 2022),
    'SG Trend, 2023–2025': ('SG Trend', 2023, 2025)})
display(attribution.round(2))

fig, ax = plt.subplots(figsize=(12, 4))
for col, key in [('pnl_trend', 'trend'), ('pnl_bias', 'bias'), ('pnl_rule', 'rule')]:
    ax.plot(pnl.index, pnl[col].rolling(104).corr(pnl['SG Trend']), color=COLORS[key], label=PNL_LABELS[col])
ax.axhline(0, color='black', lw=.8)
ax.set(title='Rolling two-year correlation of energy P&L with SG Trend Index returns', ylabel='Correlation')
ax.grid(alpha=.25)
ax.legend()
fig.tight_layout()
fig.savefig(OUTPUT / 'pnl_correlation_rolling.png', dpi=160)
plt.show()

**Reading.** Across all three SG series, trend-position P&L is the dominant and most robust explanatory variable (t between 7.2 and 8.0). Bias P&L is significant in the linear regression, but its rank correlation is near zero, and its rolling correlation swings between −0.4 and +0.55. It lines up with SG mainly when discretionary and trend positions point the same way, as in the 2022 rally. The fit residual contributes little.

This is independent evidence for the paper's labels: the component the NN calls trend-following earns money when the largest CTAs do. Two cautions:
- In 2023–2025 the trend t-statistic falls to 1.35, and from 2024 bias P&L tracks SG more closely than trend P&L. Either CTA energy behaviour shifted, or the NN's trend layer lags it.
- Only 14.7–16.9% of weekly SG variance is explained, which is expected because energy is a small part of CTA portfolios.

## Save results

In [ ]:
decomposition.to_csv(OUTPUT / 'nn_trend_bias_decomposition.csv', index=False)
pnl.to_csv(OUTPUT / 'weekly_energy_pnl_vs_sg.csv')
correlations.to_csv(OUTPUT / 'pnl_correlations.csv')
attribution.to_csv(OUTPUT / 'pnl_attribution.csv')

## References

- Sun, Bouchouev and Fattouh (2026), [Momentum Trading and Managed Money Positioning in Energy](https://www.oxfordenergy.org/wpcms/wp-content/uploads/2026/03/Insight-177-Money-Positioning-in-Energy.pdf), OIES Insight 177, §§1 and 3.
- Société Générale Prime Services, [SG Trend Index methodology](https://wholesale.banking.societegenerale.com/fileadmin/indices_feeds/SG_Trend_Index_Methodology.pdf) and [SG Trend Indicator methodology](https://wholesale.banking.societegenerale.com/fileadmin/indices_feeds/SG_Trend_Indicator_Methodology_Summary.pdf).
- CFTC, [Disaggregated COT explanatory notes](https://www.cftc.gov/MarketReports/CommitmentsofTraders/DisaggregatedExplanatoryNotes/index.htm): definition of Managed Money.
- Boos and Grob (2023), [Tracking speculative trading](https://www.sciencedirect.com/science/article/pii/S1386418122000635), *Journal of Financial Markets* 64.
- Roncalli and Teiletche (2008), [An Alternative Approach to Alternative Beta](https://doi.org/10.2139/ssrn.1035521): Kalman-filter replication of hedge fund indices.
- Benhamou, Ohana, Guez and Jacquot (2026), [Decoding CTA Allocations by Trend Horizon](https://rpc.cfainstitute.org/blogs/enterprising-investor/2026/decoding-cta-allocations-by-trend-horizon), CFA Institute.